# Biohub — labeled validation sweep

This notebook uses the competition's labeled `train/` mount and the public Biohub Tracking Support Pack. It runs the supplied UNet + temporal edge model on one training embryo, evaluates it with the supplied metric implementation, and sweeps detection thresholds.

The goal is diagnostic: separate detection quality from graph/linking quality before making another leaderboard submission.

In [ ]:
from pathlib import Path
import shutil
import os
import subprocess
import sys

INPUTS = Path('/kaggle/input')
print('attached inputs:', [p.name for p in INPUTS.iterdir()])
support_candidates = [
    INPUTS / 'biohub-tracking-support-pack-50ep-v1',
    INPUTS / 'datasets' / 'pilkwang' / 'biohub-tracking-support-pack-50ep-v1',
]
support_candidates = [p for p in support_candidates if p.exists()]
SUPPORT = support_candidates[0] if support_candidates else None
WHEELS = SUPPORT / 'wheels' if SUPPORT else None
WORK_REPO = Path('/kaggle/working/biohub_support_repo')
COMPETITION = next((p for p in INPUTS.iterdir() if p.name == 'competitions'), None)
TRAIN = (COMPETITION / 'biohub-cell-tracking-during-development' / 'train') if COMPETITION else None
assert SUPPORT is not None and SUPPORT.exists(), support_candidates
assert TRAIN.exists(), TRAIN

# Install only from the attached support-pack wheels; Internet remains disabled.
packages = ['tracksdata', 'zarr>=3.0.10,<4', 'pyscipopt', 'geff', 'ilpy', 'polars', 'blosc2', 'dask', 'imagecodecs', 'pyarrow', 'rustworkx', 'sqlalchemy']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(WHEELS), *packages])

if WORK_REPO.exists():
    shutil.rmtree(WORK_REPO)
shutil.copytree(SUPPORT / 'repo', WORK_REPO)
shutil.copytree(SUPPORT / 'weights', WORK_REPO / 'weights')
sys.path.insert(0, str(WORK_REPO / 'src'))
sys.path.insert(0, str(WORK_REPO / 'scripts'))
print('train datasets:', len(list(TRAIN.glob('*.zarr'))))
print('split file:', (TRAIN / 'dataset_splits.json').exists())
print('weights:', list((WORK_REPO / 'weights').rglob('*.pth')))

In [ ]:
# Run a one-embryo labeled validation sweep. Increase SLICE to ':5' after the
# first successful run; the full set is intentionally not run by default.
script = WORK_REPO / 'scripts' / 'predict_unet_transformer.py'
weights = WORK_REPO / 'weights' / 'unet_transformer' / 'split_0' / 'edge_predictor_best.pth'
video = sorted(TRAIN.glob('*.zarr'))[0]
run_env = os.environ.copy()
run_env['PYTHONPATH'] = str(WORK_REPO / 'src') + os.pathsep + str(WORK_REPO / 'scripts')
results = []
for threshold in ['0.99']:
    cmd = [sys.executable, str(script), '--debug-video', str(video), '--evaluate', '--det-threshold', threshold, '--weights', str(weights)]
    print('\n### threshold', threshold)
    completed = subprocess.run(cmd, cwd=WORK_REPO, env=run_env, text=True, capture_output=True)
    print(completed.stdout[-5000:])
    if completed.returncode:
        print(completed.stderr[-5000:])
    results.append({'threshold': threshold, 'returncode': completed.returncode, 'stdout': completed.stdout})
print('completed thresholds:', [r['threshold'] for r in results if r['returncode'] == 0])

## Interpretation

Record the reported `score`, `edge_jaccard`, `division_jaccard`, and `node_recall`. The next submission should use the threshold that wins on this labeled validation slice, then be checked on several more embryos. A high node recall with weak edge Jaccard points to association; weak node recall points to detection.